#  Introduction to LLM Agents

#  Part 1: Introduction to LLMs

An **LLM (Large Language Model)** is a neural network trained on massive amounts of text. Given some input text, it predicts what text should come next.

Think of it as an extremely well-read assistant that:
- Understands your question
- Draws on everything it was trained on
- Generates a relevant, coherent response

When you talk to an LLM via an API, every conversation has two key components:

| Component | Role |
|---|---|
| **System Prompt** | Sets the LLM's personality, rules, and context |
| **User Prompt** | The actual question or request |

Let's see this in action.

##  Setup

In [7]:
GPT_MODEL = "gpt-4o"

In [30]:
# !pip3.11 install openai
# !pip3.11 install dotenv


import openai
import json
import os
from dotenv import load_dotenv

print("Loading env...")
# load openai api key from .env file
load_dotenv()

print("initialising client")
# Initialize the client
client = openai.OpenAI()

print("making openai call")

response = client.chat.completions.create(
    model=GPT_MODEL,
    messages=[{"role": "user", "content": "Hello!"}]
)
print("\n-------\n")
print(response)
print(response.choices[0].message.content)

print(" Client ready!")

Loading env...
initialising client
making openai call

-------

ChatCompletion(id='chatcmpl-DTqXGjPhA5DTxPY5B9mUgpNEaheEY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1776005830, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_c6907745f9', usage=CompletionUsage(completion_tokens=9, prompt_tokens=9, total_tokens=18, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
Hello! How can I assist you today?
 Client ready!


##  System Prompt

The **system prompt** is a special instruction block that shapes HOW the model behaves — its tone, its constraints, its persona.

It is set once per conversation and the user typically never sees it.

**Key uses:**
- Give the model a role (`"You are a helpful customer support agent for Acme Corp."`)
- Set hard rules (`"Never discuss competitor products."`)
- Provide background knowledge (`"Our return policy is 30 days."`)
- Define output format (`"Always respond in bullet points."`)

In [31]:
# ── System Prompt Demo ──────────────────────────────────────────────

system_prompt = """
You are a friendly and concise Python tutor for beginners.
- Use simple language. Avoid jargon.
- give your answer in bullet polints
- Always give a short code example.
- Keep answers under 5 sentences.
"""

user_prompt = "What is a list in Python?"


response = client.chat.completions.create(
    model=GPT_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},   # <--- system prompt
        {"role": "user", "content": user_prompt},        # <--- user prompt}
    ]
)

print(response.choices[0].message.content)

A list in Python is a collection that can hold multiple items, which can be of different types, such as numbers, strings, or even other lists. Lists are ordered, changeable, and allow duplicate values. You create a list using square brackets.

Here’s a simple example of a list:

```python
fruits = ["apple", "banana", "cherry"]
```

In this example, `fruits` is a list containing three string items.


##  User Prompt

The **user prompt** is what the user actually types — the question, task, or instruction they want the model to handle.

In a multi-turn conversation, you send the full history of user/assistant messages so the model has context.

> **Key insight:** The model has NO memory between separate API calls. You must pass the conversation history yourself every time.

In [17]:
# ── Multi-turn conversation: simulating memory ─────────────────────

conversation_history = []

def chat(user_message):
    """Send a message and maintain conversation history."""
    # Add the new user message to history
    conversation_history.append({"role": "user", "content": user_message})
    
    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=conversation_history  # Send the FULL history each time
    )
    
    assistant_reply = response.choices[0].message.content
    
    # Add the assistant's reply to history so next turn has context
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Turn 1
print("Assistant:", chat("My name is Alex."))


# Turn 2 — model should remember the name from Turn 1
print("Assistant:", chat("What's my name?"))

Assistant: Hello, Alex! How can I assist you today?
Assistant: Your name is Alex.


#  Part 2: LLM vs Agent

## The Core Distinction

> **An LLM only generates text. An Agent takes actions.**

| | LLM | Agent |
|---|---|---|
| **What it does** | Generates text | Generates text + calls tools |
| **Can search the web?** | ❌ No | ✅ Yes (with a search tool) |
| **Can send emails?** | ❌ No | ✅ Yes (with an email tool) |
| **Can run code?** | ❌ No | ✅ Yes (with a code tool) |
| **Has memory of actions?** | ❌ No | ✅ Yes (within a session) |

## The Analogy

```
LLM   =  Brain alone
             - Brilliant, knowledgeable
             - But can only THINK, not DO

Agent =  Brain(LLM to think and plan) + Body(Tool calling + Execution)
             - Can think (LLM)
             - Can act (tools: search, APIs, databases, code)
             - Can perceive results and decide next steps
```

## How an Agent Loop Works

```
User Input
    │
    ▼
┌─────────────────────────────┐
│           LLM               │
│  "What should I do next?"   │
└──────────┬──────────────────┘
           │
    ┌──────▼──────┐
    │  Tool Call  │  ← e.g., search("weather in Hyderabad")
    └──────┬──────┘
           │
    ┌──────▼──────┐
    │Tool Response│  ← "Sunny, 34°C"
    └──────┬──────┘
           │
    ┌──────▼──────────────────┐
    │   LLM again             │
    │ "Now I can answer!"      │
    └─────────────────────────┘
           │
    Final Answer to User
```

The key insight: **The LLM decides WHEN to call a tool and WHICH tool to call.** The developer defines what tools exist. The model reasons about which ones to use.

#  Part 3: Tool Calling

## What is Tool Calling?

Tool calling (also called **function calling**) is the mechanism that turns an LLM into an agent.

Here's how it works:
1. **You define tools** — describe them in JSON (name, description, parameters)
2. **User sends a message** — e.g., *"What's the weather in Hyderabad?"*
3. **LLM decides** — *"I need to call `get_weather` with `city='Hyderabad'`"*
4. **You execute the tool** — your Python code actually calls the weather API
5. **You return the result** — send the result back to the LLM
6. **LLM formulates a final answer** — using the real data

> The LLM never directly calls your functions. It outputs a **structured request** saying "please call this function with these arguments". Your code does the actual execution.

## Step-by-Step: Your First Tool Call

Let's build a simple calculator tool so the LLM can do arithmetic accurately.

In [21]:
# ── Step 1: Define the tool (what the LLM can call) ─────────────────


tools = [
    {
        "type": "function",  
        "function": {
            "name": "calculate",
            "description": "Evaluates a mathematical expression and returns the result.",
            "parameters": {  
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A valid Python math expression like '2 + 2'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("Tool defined!")

Tool defined!


In [22]:
# ── Step 2: Define the actual Python function that runs the tool ─────

def calculate(expression: str) -> str:
    """Safely evaluate a math expression."""
    try:
        result = eval(expression, {"__builtins__": {}})  # Restricted eval
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Test it directly
print(calculate("123 * 456"))  # 56088

56088


In [26]:
# ── Step 3: Send user message to LLM with tools available ────────────

user_message = "What is 123 multiplied by 456?"

response = client.chat.completions.create(
    model=GPT_MODEL,
    tools=tools,
    messages=[
        {"role": "user", "content": user_message}
    ]
)

print(response)


ChatCompletion(id='chatcmpl-DTpUmTvJG3gQBMFsBrvhGQlctd0BZ', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cOPGs1Gc5bheayHlLPk0r4ty', function=Function(arguments='{"expression":"123 * 456"}', name='calculate'), type='function')]))], created=1776001832, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_60fa608354', usage=CompletionUsage(completion_tokens=16, prompt_tokens=68, total_tokens=84, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [27]:
# ── Step 4: Handle the tool call  ────────────────────

# A dispatcher: maps tool names to actual Python functions
tool_registry = {
    "calculate": calculate
}

def run_agent(user_message: str, tools: list, system: str = "") -> str:
    """A simple agent loop: keep going until the LLM gives a final answer."""
    
    messages = []
    
    if system:
        messages.append({"role": "system", "content": system})
    
    messages.append({"role": "user", "content": user_message})
    
    while True:
        response = client.chat.completions.create(
            model=GPT_MODEL,
            tools=tools,
            messages=messages
        )
        
        message = response.choices[0].message
        
        # LLM is done — return the final text response
        if not message.tool_calls:
            return message.content
        
        # LLM wants to call a tool
        
        # Add the LLM's response (with tool call request) to history
        messages.append(message)
        
        # Process each tool call in the response
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            
            print(f"   Calling tool: {tool_name}({tool_input})")
            
            # Actually run the tool
            fn = tool_registry[tool_name]
            result = fn(**tool_input)
            
            print(f"   Tool result: {result}")
            
            # Add tool result back to history so LLM can see it
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        # Loop again — LLM will now formulate its final answer


# Run it!
answer = run_agent("What is 123 multiplied by 456? Also, what is 999 + 1?", tools)
print("\n Final Answer:", answer)

   Calling tool: calculate({'expression': '123 * 456'})
   Tool result: 56088
   Calling tool: calculate({'expression': '999 + 1'})
   Tool result: 1000

 Final Answer: The result of 123 multiplied by 456 is 56,088, and the result of 999 plus 1 is 1,000.


## 🧪 Try it yourself — Add a new tool!

Let's add a `string_reverse` tool and see the agent use it.

In [28]:
# ── Exercise: add a new tool (OpenAI style) ─────────────────────────

# 1. Define the tool schema
tools_v2 = tools + [
    {
        "type": "function",
        "function": {
            "name": "string_reverse",
            "description": "Reverses a given string. Use when the user asks to reverse text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "The string to reverse"
                    }
                },
                "required": ["text"]
            }
        }
    }
]

# 2. Implement the function
def string_reverse(text: str) -> str:
    return text[::-1]

# 3. Register it
tool_registry["string_reverse"] = string_reverse

# 4. Test!
answer = run_agent("Reverse the word 'Hyderabad' and also compute 50 * 50.", tools_v2)
print("\n Final Answer:", answer)

   Calling tool: string_reverse({'text': 'Hyderabad'})
   Tool result: dabaredyH
   Calling tool: calculate({'expression': '50 * 50'})
   Tool result: 2500

 Final Answer: The reverse of the word 'Hyderabad' is 'dabaredyH', and the result of \(50 \times 50\) is 2500.


#  Take-Home Project: Password Reset Chatbot

## The Mission

Build a chatbot that can **reset a user's password**. The agent should:

1. Ask the user for their **SSN** (Social Security / ID number)
2. Ask for their **old password** and the **new password** they want
3. Call a `change_password` tool with those details
4. Print **"Password Changed!"** if successful


## Starter Code

Follow the steps below. Each cell has hints for what you need to fill in.

In [29]:
# ── STEP 1: Fake User Database ────────────────────────────────────────
# In a real system, this would be a database. For now, a dictionary is fine.

USER_DB = {
    "123-45-6789": {"password": "oldpass123", "name": "Alice"},
    "987-65-4321": {"password": "securepass!", "name": "Bob"},
}

print("User DB loaded with", len(USER_DB), "users.")

User DB loaded with 2 users.


In [ ]:
# ── STEP 2: Implement the change_password function ────────────────────
# This is the actual logic that changes the password.
# The LLM will call this via tool_use.

def change_password(ssn: str, old_password: str, new_password: str) -> str:
    """
    Attempts to change the password for the given SSN.
    Returns 'Password Changed!' on success, or an error message.
    """
    # TODO: Implement this function!
    


# Quick test (uncomment after implementing):
# print(change_password("123-45-6789", "oldpass123", "newpass456"))  # Should print: Password Changed!
# print(change_password("123-45-6789", "wrongpass", "newpass456"))   # Should print: Error message
# print(change_password("000-00-0000", "anything", "anything"))      # Should print: Error message

In [ ]:
# ── STEP 3: Define the tool schem and register ────────────────────────────────────


In [ ]:
# ── STEP 4: Write the System Prompt ───────────────────────────────────



In [ ]:
# ── STEP 5: Build the Chatbot Loop ────────────────────────────────────
# This is the interactive chatbot. It runs a conversation in a loop.

